In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import numpy as np
import tensorflow as tf
import os
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from data_process import Anekdoot
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm import tqdm
import matplotlib.ticker as mticker
import seaborn as sns
import pickle
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

d:\Repositories\MA_1y_1s_HW\venv3.12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**BERT initialization**

In [ ]:
#EstBERT model
normal_tokenizer = BertTokenizer.from_pretrained('tartuNLP/EstBERT')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9627.84it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

In [ ]:
# Initialize data
file_path = "anekdoodid_with_types_inimene.csv"

an = Anekdoot(file_path=file_path)

dataframe = an.read_dataframe()
X_anecdotes = an.extract_main_col(dataframe, "Anekdoot").tolist() # We only want the anecdotes for training
y_anecdotes = an.extract_main_col(dataframe, "anekdoodi tüüp").tolist() # Labels for training

# Filter out samples with NaN labels (both actual NaN and literal "nan" string)
import pandas as pd
mask = [pd.notna(y) and str(y).strip().lower() != "nan" for y in y_anecdotes]
X_anecdotes = [x for x, m in zip(X_anecdotes, mask) if m]
y_anecdotes = [y for y, m in zip(y_anecdotes, mask) if m]

print(f"After removing NaN labels: {len(X_anecdotes)} samples")

# --- Balanced sampling: soft-cap per class ---
def balanced_sample(X, y, n_per_class=None, random_state=42):
    """
    Return a class-balanced subset of (X, y).
    n_per_class: target samples per class. Classes with fewer samples than
    n_per_class use all their available data instead of being excluded.
    If None, uses the minimum class count (hard-balanced).
    """
    import random
    rng = random.Random(random_state)

    # Group indices by class
    class_indices = {}
    for idx, label in enumerate(y):
        class_indices.setdefault(label, []).append(idx)

    # Print class distribution before sampling
    print("\nClass distribution (full dataset):")
    for cls, idxs in sorted(class_indices.items()):
        print(f"  {cls}: {len(idxs)}")

    if n_per_class is None:
        n_per_class = min(len(v) for v in class_indices.values())

    print(f"\nTarget {n_per_class} samples per class (soft-cap: smaller classes use all available):")
    sampled_X, sampled_y = [], []
    for label, idxs in sorted(class_indices.items()):
        take = min(n_per_class, len(idxs))
        chosen = rng.sample(idxs, take)
        print(f"  {label}: {take}")
        for i in chosen:
            sampled_X.append(X[i])
            sampled_y.append(y[i])

    total = len(sampled_X)
    print(f"\n{len(class_indices)} classes → {total} total samples")

    # Shuffle the combined list
    combined = list(zip(sampled_X, sampled_y))
    rng.shuffle(combined)
    sampled_X, sampled_y = zip(*combined)
    return list(sampled_X), list(sampled_y)

# n_per_class=200: classes with fewer samples (e.g. kuldkala=42, ämm=77) use all
# their available data; all other classes are capped at 200.
X_anecdotes_subset, y_anecdotes_subset = balanced_sample(
    X_anecdotes, y_anecdotes, n_per_class=200, random_state=42
)

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_anecdotes_subset)
num_labels = len(label_encoder.classes_)

print(f"\nClasses: {label_encoder.classes_}")
print(f"Number of classes: {num_labels}")
print(f"Dataset size: {len(X_anecdotes_subset)}")

# Tokenize all texts with proper padding/truncation
encoded_inputs = normal_tokenizer(
    X_anecdotes_subset,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors='pt'
)

print(f"Input IDs shape: {encoded_inputs['input_ids'].shape}")
print(f"Attention mask shape: {encoded_inputs['attention_mask'].shape}")

After removing NaN labels: 24053 samples

Class distribution (full dataset):
  blondiin: 992
  ema: 577
  isa: 414
  joodik: 353
  kalamees: 293
  kiriku: 906
  kool: 1636
  korrakaitse: 559
  kuldkala: 42
  loomad: 2605
  meditsiin: 1773
  mees: 2287
  mees ja naine: 293
  militaar: 499
  müller stirlitz: 208
  naine: 3196
  poliitika: 1043
  rahvus: 1696
  riigianekdoodid: 1628
  roppused: 2524
  tsuks ja petka: 227
  vangla: 225
  ämm: 77

Sampling 42 samples per class (23 classes → 966 total)

Classes: ['blondiin' 'ema' 'isa' 'joodik' 'kalamees' 'kiriku' 'kool' 'korrakaitse'
 'kuldkala' 'loomad' 'meditsiin' 'mees' 'mees ja naine' 'militaar'
 'müller stirlitz' 'naine' 'poliitika' 'rahvus' 'riigianekdoodid'
 'roppused' 'tsuks ja petka' 'vangla' 'ämm']
Number of classes: 23
Dataset size: 966
Input IDs shape: torch.Size([966, 128])
Attention mask shape: torch.Size([966, 128])


In [ ]:

# Define Dataset class
class AnecdoteDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Split data into train/validation with stratification to ensure even class distribution
X_train, X_val, y_train, y_val = train_test_split(
    X_anecdotes_subset, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Create datasets
train_dataset = AnecdoteDataset(X_train, y_train, normal_tokenizer)
val_dataset = AnecdoteDataset(X_val, y_val, normal_tokenizer)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Number of classes: {num_labels}")

Training samples: 772
Validation samples: 194
Number of classes: 23


In [ ]:
from transformers import get_linear_schedule_with_warmup
import warnings

# Training setup - Auto-detect best available device
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"Using NVIDIA CUDA GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch, 'xpu') and torch.xpu.is_available():
    device = torch.device('xpu')
    print(f"Using Intel XPU GPU")
else:
    device = torch.device('cpu')
    warnings.warn(
        "No GPU detected - training will run on CPU and be very slow.\n"
        "If you are on Google Colab: Runtime → Change runtime type → set Hardware accelerator to T4 GPU → Save, then reconnect.",
        RuntimeWarning
    )
    print("Using CPU (no GPU detected)")

print(f"Device: {device}")

# Load EstBERT — weights now match the tokenizer loaded in cell 2
bert_model = BertForSequenceClassification.from_pretrained(
    'tartuNLP/EstBERT',
    num_labels=num_labels
).to(device)

epochs = 10
optimizer = AdamW(bert_model.parameters(), lr=2e-5)

# Linear warmup for first 10% of steps, then linear decay to 0
total_steps = len(train_loader) * epochs
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# Checkpoint directory
checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
best_loss = float('inf')
epoch_losses = []  # collected for training loss curve plot

# Training loop
bert_model.train()
for epoch in range(epochs):
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')

    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = bert_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        progress_bar.set_postfix({'loss': loss.item()})

    avg_loss = total_loss / len(train_loader)
    epoch_losses.append(avg_loss)
    print(f"Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")

    # Save checkpoint if this epoch has the best loss so far
    if avg_loss < best_loss:
        best_loss = avg_loss
        checkpoint_path = os.path.join(checkpoint_dir, "best_model.pt")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': bert_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': best_loss,
            'epoch_losses': epoch_losses,
            'label_encoder_classes': label_encoder.classes_,
        }, checkpoint_path)
        print(f"  → Saved best checkpoint (loss={best_loss:.4f}) to {checkpoint_path}")

print("Training complete!")

Using CPU (no GPU detected)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18857.83it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider t

Epoch 1 - Average Loss: 3.1610


Epoch 2/10: 100%|██████████| 49/49 [01:48<00:00,  2.21s/it, loss=3.32]


Epoch 2 - Average Loss: 3.0946


...

Epoch 8 - Average Loss: 1.5645


Epoch 9/10: 100%|██████████| 49/49 [01:48<00:00,  2.21s/it, loss=0.652]


Epoch 9 - Average Loss: 1.2224


Epoch 10/10: 100%|██████████| 49/49 [01:47<00:00,  2.20s/it, loss=0.385]

Epoch 10 - Average Loss: 0.9581
Training complete!


In [6]:
# Evaluation
bert_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = bert_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate accuracy
accuracy = accuracy_score(all_labels, all_preds)
print(f"Validation Accuracy: {accuracy:.4f}")

# Get unique classes that actually appear in the data
unique_labels = sorted(set(all_labels) | set(all_preds))
target_names_used = [label_encoder.classes_[i] for i in unique_labels]

# Detailed classification report
print("\nClassification Report:")
print(classification_report(
    all_labels, 
    all_preds, 
    labels=unique_labels,
    target_names=target_names_used
))

Validation Accuracy: 0.5979

Classification Report:
                 precision    recall  f1-score   support

       blondiin       0.91      1.00      0.95        10
            ema       1.00      0.67      0.80         9
            isa       1.00      0.71      0.83         7
         joodik       0.14      0.14      0.14         7
       kalamees       1.00      0.75      0.86         8
         kiriku       0.00      0.00      0.00         7
           kool       0.62      0.56      0.59         9
    korrakaitse       0.80      0.89      0.84         9
       kuldkala       0.90      1.00      0.95         9
         loomad       0.27      0.27      0.27        11
      meditsiin       0.40      0.29      0.33         7
           mees       0.00      0.00      0.00         7
  mees ja naine       0.45      1.00      0.62         5
       militaar       0.50      0.50      0.50        10
müller stirlitz       1.00      1.00      1.00         7
          naine       0.83      0.6

In [ ]:

class_names = label_encoder.classes_

# ── 1. Training loss curve ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(epoch_losses) + 1), epoch_losses, marker='o', color='steelblue', linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Average Training Loss")
ax.set_title("BERT — Training Loss per Epoch")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("bert_training_loss.png", dpi=150)
plt.show()
print("Saved: bert_training_loss.png")

# ── 2. Confusion matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names,
    linewidths=0.4, ax=ax
)
ax.set_xlabel("Predicted label", fontsize=12)
ax.set_ylabel("True label", fontsize=12)
ax.set_title("BERT — Confusion Matrix (validation set)", fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig("bert_confusion_matrix.png", dpi=150)
plt.show()
print("Saved: bert_confusion_matrix.png")

# ── 3. Per-class F1 bar chart ─────────────────────────────────────────────────
report = classification_report(all_labels, all_preds, output_dict=True, zero_division=0)
f1_scores = [report[str(i)]['f1-score'] for i in range(len(class_names))]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(class_names, f1_scores, color='steelblue', edgecolor='white')
ax.set_ylim(0, 1.05)
ax.set_ylabel("F1-score")
ax.set_title("BERT — Per-class F1-score (validation set)")
plt.xticks(rotation=45, ha='right', fontsize=9)
ax.axhline(np.mean(f1_scores), color='firebrick', linestyle='--', linewidth=1.5, label=f'Macro avg F1 = {np.mean(f1_scores):.3f}')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("bert_per_class_f1.png", dpi=150)
plt.show()
print("Saved: bert_per_class_f1.png")

# ── 4. Summary metrics ────────────────────────────────────────────────────────
macro_f1  = report['macro avg']['f1-score']
weighted_f1 = report['weighted avg']['f1-score']
val_acc = accuracy_score(all_labels, all_preds)
print(f"\nBERT Summary")
print(f"  Validation accuracy : {val_acc:.4f}")
print(f"  Macro F1            : {macro_f1:.4f}")
print(f"  Weighted F1         : {weighted_f1:.4f}")

In [ ]:

# ── Reload baseline artifacts ─────────────────────────────────────────────────
baseline_dir = "baseline_export"
with open(f"{baseline_dir}/tfidf_vectorizer.pkl", "rb") as f:
    bl_vectorizer = pickle.load(f)
with open(f"{baseline_dir}/logistic_regression.pkl", "rb") as f:
    bl_classifier = pickle.load(f)
with open(f"{baseline_dir}/label_encoder.pkl", "rb") as f:
    bl_label_encoder = pickle.load(f)

# Re-produce the same val split used in the baseline notebook
# (identical parameters: same X_anecdotes_subset, y_encoded, random_state, stratify)
from sklearn.model_selection import train_test_split
_, X_val_bl, _, y_val_bl = train_test_split(
    X_anecdotes_subset, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
X_val_tfidf = bl_vectorizer.transform(X_val_bl)
y_pred_bl = bl_classifier.predict(X_val_tfidf)

class_names = label_encoder.classes_

# ── Reports for both models ───────────────────────────────────────────────────
report_bert = classification_report(all_labels, all_preds,    output_dict=True, zero_division=0)
report_bl   = classification_report(y_val_bl,   y_pred_bl,   output_dict=True, zero_division=0)

f1_bert = np.array([report_bert[str(i)]['f1-score'] for i in range(len(class_names))])
f1_bl   = np.array([report_bl  [str(i)]['f1-score'] for i in range(len(class_names))])

# ── Plot 1: Side-by-side per-class F1 ────────────────────────────────────────
x = np.arange(len(class_names))
width = 0.38

fig, ax = plt.subplots(figsize=(15, 6))
ax.bar(x - width/2, f1_bert, width, label='BERT (EstBERT)', color='steelblue',  edgecolor='white')
ax.bar(x + width/2, f1_bl,   width, label='Baseline (TF-IDF + LR)', color='darkorange', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=9)
ax.set_ylim(0, 1.1)
ax.set_ylabel("F1-score")
ax.set_title("Per-class F1-score — BERT vs Baseline")
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("comparison_per_class_f1.png", dpi=150)
plt.show()
print("Saved: comparison_per_class_f1.png")

# ── Plot 2: Side-by-side confusion matrices ───────────────────────────────────
cm_bert = confusion_matrix(all_labels, all_preds)
cm_bl   = confusion_matrix(y_val_bl,   y_pred_bl)

fig, axes = plt.subplots(1, 2, figsize=(26, 11))
for ax, cm, title, cmap in zip(
    axes,
    [cm_bert, cm_bl],
    ["BERT (EstBERT)", "Baseline (TF-IDF + LR)"],
    ["Blues", "Oranges"]
):
    sns.heatmap(
        cm, annot=True, fmt='d', cmap=cmap,
        xticklabels=class_names, yticklabels=class_names,
        linewidths=0.3, ax=ax
    )
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Predicted label", fontsize=11)
    ax.set_ylabel("True label", fontsize=11)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

plt.suptitle("Confusion Matrices — BERT vs Baseline (validation set)", fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig("comparison_confusion_matrices.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: comparison_confusion_matrices.png")

# ── Plot 3: Summary bar chart (accuracy, macro F1, weighted F1) ───────────────
metrics = ['Accuracy', 'Macro F1', 'Weighted F1']
bert_vals = [
    accuracy_score(all_labels, all_preds),
    report_bert['macro avg']['f1-score'],
    report_bert['weighted avg']['f1-score'],
]
bl_vals = [
    accuracy_score(y_val_bl, y_pred_bl),
    report_bl['macro avg']['f1-score'],
    report_bl['weighted avg']['f1-score'],
]

x = np.arange(len(metrics))
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(x - 0.2, bert_vals, 0.38, label='BERT (EstBERT)',       color='steelblue',  edgecolor='white')
ax.bar(x + 0.2, bl_vals,   0.38, label='Baseline (TF-IDF+LR)', color='darkorange', edgecolor='white')
for i, (bv, bv2) in enumerate(zip(bert_vals, bl_vals)):
    ax.text(i - 0.2, bv + 0.01, f'{bv:.3f}', ha='center', va='bottom', fontsize=9)
    ax.text(i + 0.2, bv2 + 0.01, f'{bv2:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score")
ax.set_title("Overall metrics — BERT vs Baseline")
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("comparison_summary_metrics.png", dpi=150)
plt.show()
print("Saved: comparison_summary_metrics.png")

In [ ]:
import pickle
import json

# Export BERT model in HuggingFace format for post-hoc analysis (SHAP / LIME / attention)

export_dir = "bert_export"
os.makedirs(export_dir, exist_ok=True)

# 1. Save the fine-tuned model weights + config in HuggingFace format
bert_model.save_pretrained(export_dir)
print(f"Model saved to {export_dir}/")

# 2. Save the tokenizer so analysis scripts can reproduce the exact tokenisation
normal_tokenizer.save_pretrained(export_dir)
print(f"Tokenizer saved to {export_dir}/")

# 3. Save label encoder classes as JSON for easy reuse in analysis scripts
label_map = {int(i): str(cls) for i, cls in enumerate(label_encoder.classes_)}
with open(os.path.join(export_dir, "label_map.json"), "w", encoding="utf-8") as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)
print(f"Label map saved to {export_dir}/label_map.json")

# 4. Save the label encoder itself (needed if analysis code uses sklearn)
with open(os.path.join(export_dir, "label_encoder.pkl"), "wb") as f:
    pickle.dump(label_encoder, f)
print(f"Label encoder saved to {export_dir}/label_encoder.pkl")

print("\n--- Reload example (use this in your analysis notebook) ---")
print(f"""
from transformers import BertTokenizer, BertForSequenceClassification
import json, pickle

model = BertForSequenceClassification.from_pretrained("{export_dir}")
tokenizer = BertTokenizer.from_pretrained("{export_dir}")

with open("{export_dir}/label_map.json") as f:
    label_map = json.load(f)   # {{0: 'blondiin', 1: 'ema', ...}}
""")